# Notebook 05 — Contact Maps and Distance Geometry

**Module:** 11 — Structural Biology  
**Tier:** 3 — Survey  
**Notebook:** 05 of 08  
**Time estimate:** 60 minutes

> A contact map encodes 3D structure as a 2D binary matrix.
> It is the bridge between sequence co-evolution (from MSAs) and
> 3D structure prediction — the conceptual foundation of AlphaFold2.

---
## Step 1 — Motivation

AlphaFold2's key intermediate representation is a pair representation — essentially
an evolved contact map. Understanding contact maps explains what AlphaFold2's
Evoformer is learning to predict before the structure module constructs 3D coordinates.

---
## Step 2 — Intuition

**Contact map:** For a protein of $N$ residues, the contact map is an $N \times N$
symmetric binary matrix where entry $(i, j) = 1$ if residues $i$ and $j$ are in
contact (Cβ distance ≤ 8 Å) and 0 otherwise.

The contact map is a 2D fingerprint of the 3D structure:
- The diagonal band = sequential neighbours (always in contact)
- Off-diagonal blocks = long-range contacts (fold topology)
- Pattern of blocks = secondary structure elements

**Co-evolution connection:** If residue $i$ and residue $j$ are in contact in the
3D structure, mutations at position $i$ are constrained by the amino acid at $j$
(they must be compatible). This co-evolutionary signal in multiple sequence alignments
was the basis of direct coupling analysis (DCA) — and is what AlphaFold2's Evoformer
learns to extract more powerfully.

---
## Step 3 — Biological Background

**Why Cβ and not Cα?**  
Cα is the backbone atom; Cβ is the first side-chain atom. Contacts between Cβ atoms
reflect side-chain interactions, which are more relevant to binding and stability.
For glycine (no Cβ), Cα is used instead.

**The 8 Å threshold:**  
Standard choice for residue contacts — covers most van der Waals and electrostatic
interactions while avoiding noise from too-distant pairs. Some methods use 6 Å
(stricter) or 12 Å (more inclusive).

**Contact map patterns:**
- **α-helix contacts:** diagonal stripe, contacts at $|i-j|$ = 3 and 4
- **β-sheet (parallel):** off-diagonal stripe (i~j+k for large k)
- **β-sheet (antiparallel):** antidiagonal stripe
- **Domain interfaces:** distinct block pattern

**Distance geometry:**  
Given a distance matrix $D$ where $D_{ij}$ = distance between residues $i$ and $j$,
can we recover 3D coordinates? Yes, via **metric matrix embedding** (classical MDS).
This is the basis of early structure prediction from contact predictions.

---
## Step 4 — Mathematical Explanation

**Distance matrix from coordinates:**
$$D_{ij} = \|\mathbf{r}_i - \mathbf{r}_j\|$$

Efficient computation via broadcasting:
$$D_{ij}^2 = \|\mathbf{r}_i\|^2 + \|\mathbf{r}_j\|^2 - 2\mathbf{r}_i \cdot \mathbf{r}_j$$

**Contact map from distance matrix:**
$$C_{ij} = \begin{cases} 1 & D_{ij} \leq d_{\text{thresh}} \\ 0 & \text{otherwise} \end{cases}$$

**Classical MDS (distance geometry to coordinates):**  
1. Double-centre the squared distance matrix:
   $B_{ij} = -\frac{1}{2}(D_{ij}^2 - \bar{D}_{i\cdot}^2 - \bar{D}_{\cdot j}^2 + \bar{D}_{\cdot\cdot}^2)$
2. Eigendecompose $B = V \Lambda V^T$
3. Coordinates: $X = V_{:,1:k} \Lambda_{1:k}^{1/2}$ (take top-$k$ components)

This recovers the original coordinates (up to rotation/reflection) from the distance
matrix — provided all distances are exact.

In [ ]:
# Step 6 — Python: Contact map from scratch + MDS distance geometry
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(42)

# ---- Generate a synthetic protein-like Cβ trace ----
# Mimic an α/β protein: N-terminal helix + sheet region
def generate_mixed_structure(n_helix=40, n_sheet=40, n_loop=20, rng=None):
    """Generate Cβ coordinates for a mixed helix+sheet protein."""
    if rng is None:
        rng = np.random.default_rng(0)
    coords = []
    # Helix segment
    for i in range(n_helix):
        angle = 2 * np.pi * i / 3.6
        x = 2.3 * np.cos(angle) + rng.normal(0, 0.2)
        y = 2.3 * np.sin(angle) + rng.normal(0, 0.2)
        z = i * 1.5 + rng.normal(0, 0.1)
        # Cβ offset from Cα (approx 1.5 Å)
        coords.append([x + 1.2, y + 0.5, z])
    # Sheet segment (antiparallel two strands)
    offset_y = 12.0  # helix ends, sheet begins at offset
    for i in range(n_sheet // 2):
        z = i * 3.3
        coords.append([0.0 + rng.normal(0, 0.2), offset_y,     z])
    for i in range(n_sheet // 2, 0, -1):
        z = i * 3.3
        coords.append([5.0 + rng.normal(0, 0.2), offset_y + 5, z])
    # Loop (random walk)
    last = np.array(coords[-1])
    for _ in range(n_loop):
        step = rng.normal(0, 2, 3)
        last = last + step / np.linalg.norm(step) * 3.8
        coords.append(last.tolist())
    return np.array(coords)

cb_coords = generate_mixed_structure(n_helix=40, n_sheet=40, n_loop=20, rng=rng)
N = len(cb_coords)
print(f'Total residues: {N}')

# ---- Compute Cβ distance matrix ----
def compute_distance_matrix(coords):
    """Pairwise Euclidean distance matrix via broadcasting."""
    diff = coords[:, np.newaxis, :] - coords[np.newaxis, :, :]  # (N, N, 3)
    return np.sqrt((diff**2).sum(axis=-1))

D = compute_distance_matrix(cb_coords)

# ---- Contact map ----
def contact_map(D, threshold=8.0, min_seqsep=5):
    """Binary contact map: 1 if distance ≤ threshold and |i-j| ≥ min_seqsep."""
    C = (D <= threshold).astype(int)
    # Zero out near-diagonal (sequential neighbours)
    for k in range(-min_seqsep + 1, min_seqsep):
        np.fill_diagonal(C[max(0, -k):, max(0, k):], 0)
    return C

C = contact_map(D, threshold=8.0)
n_contacts = C.sum() // 2  # symmetric matrix
print(f'Number of contacts (d≤8Å, |i-j|≥5): {n_contacts}')

# ---- MDS: recover 3D coordinates from distance matrix ----
def mds_from_distances(D, n_components=3):
    """Classical MDS to recover coordinates from a distance matrix."""
    n = D.shape[0]
    D2 = D**2
    # Double centering
    row_mean = D2.mean(axis=1, keepdims=True)
    col_mean = D2.mean(axis=0, keepdims=True)
    total_mean = D2.mean()
    B = -0.5 * (D2 - row_mean - col_mean + total_mean)
    # Eigendecomposition
    vals, vecs = np.linalg.eigh(B)
    # Sort descending
    idx = np.argsort(vals)[::-1]
    vals, vecs = vals[idx], vecs[:, idx]
    # Take top n_components (clip negative eigenvalues)
    vals_pos = np.maximum(vals[:n_components], 0)
    X = vecs[:, :n_components] * np.sqrt(vals_pos)[np.newaxis, :]
    return X

X_recovered = mds_from_distances(D, n_components=3)
# Compare recovered vs. original via RMSD (after centring)
cb_centred = cb_coords - cb_coords.mean(axis=0)
X_centred  = X_recovered - X_recovered.mean(axis=0)
# Note: MDS gives arbitrary rotation; compute just scale check
D_recovered = compute_distance_matrix(X_recovered)
diff_D = np.abs(D - D_recovered)
print(f'\nMDS reconstruction: mean |D_orig - D_recovered| = {diff_D.mean():.4f} Å')
print(f'(Should be ~0 for exact distances)')

In [ ]:
# Step 7 — Visualization: Distance matrix, contact map, MDS reconstruction
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Panel A: Cβ distance matrix
ax = axes[0]
im = ax.imshow(D, cmap='viridis_r', aspect='auto', vmax=30)
plt.colorbar(im, ax=ax, label='Cβ distance (Å)')
ax.set_xlabel('Residue index')
ax.set_ylabel('Residue index')
ax.set_title('A. Cβ distance matrix\n(dark = close, light = far)')
# Add region labels
ax.axhline(40, color='white', ls='--', lw=0.8, alpha=0.7)
ax.axvline(40, color='white', ls='--', lw=0.8, alpha=0.7)
ax.axhline(80, color='white', ls='--', lw=0.8, alpha=0.7)
ax.axvline(80, color='white', ls='--', lw=0.8, alpha=0.7)
ax.text(20, N-3, 'Helix', color='white', fontsize=7)
ax.text(55, N-3, 'Sheet', color='white', fontsize=7)
ax.text(87, N-3, 'Loop', color='white', fontsize=7)

# Panel B: Contact map
ax = axes[1]
ax.imshow(C, cmap='Blues', aspect='auto', interpolation='none')
ax.set_xlabel('Residue index')
ax.set_ylabel('Residue index')
ax.set_title(f'B. Contact map (threshold=8Å)\n{n_contacts} contacts, |i-j|≥5')

# Panel C: MDS reconstruction (top 2 components)
ax = axes[2]
from mpl_toolkits.mplot3d import Axes3D
colors_seq = plt.cm.plasma(np.linspace(0, 1, N))
for k in range(N - 1):
    ax.plot([X_recovered[k, 0], X_recovered[k+1, 0]],
             [X_recovered[k, 1], X_recovered[k+1, 1]],
             '-', color=colors_seq[k], lw=1.2, alpha=0.8)
sc = ax.scatter(X_recovered[:, 0], X_recovered[:, 1],
                  c=range(N), cmap='plasma', s=15, alpha=0.9, zorder=3)
plt.colorbar(sc, ax=ax, label='Residue index')
ax.set_xlabel('MDS dim 1')
ax.set_ylabel('MDS dim 2')
ax.set_title('C. MDS reconstruction (2D)\nfrom exact distance matrix')

plt.suptitle('Module 11 NB05: Contact Maps and Distance Geometry', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('contact_maps.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Step 8 — Exercises

1. Implement `compute_distance_matrix(coords)` using broadcasting only — no loops.
2. Vary the contact threshold from 6 to 12 Å. How does the number of contacts
   change? Plot contacts vs. threshold.
3. What pattern do you expect to see in the contact map for a purely α-helical
   protein vs. a purely β-sheet protein? Draw it schematically.
4. (Survey) What is direct coupling analysis (DCA) and how does it extract
   contact information from a multiple sequence alignment?

---
## Step 10 — Quiz

1. What does entry $(i, j) = 1$ in a contact map mean?
2. Why is the contact map symmetric?
3. Why do we use Cβ rather than Cα for contact maps?
4. What is the role of contact maps in the pre-AlphaFold2 structure prediction era?
5. What does MDS recover from a distance matrix? What are its limitations for
   real contact maps (where only some contacts are known)?

---
## Step 12 — Reflection

> *[In one paragraph: explain the connection between evolutionary co-variation
> in a multiple sequence alignment and contact maps, and why this connection
> was central to the development of structure prediction before AlphaFold2.]*

---
*Next: `06_alphafold2_and_protein_folding.ipynb`*